# Multi-Modal BiLSTM Yelp Fake Detector — **FOCAL LOSS** variant of Model B (tuned)

Same multi-modal architecture as Model B's LSTM (GloVe+BiLSTM text ⊕ MLP behavioural → fusion →
sigmoid), trained on the **pure full** Yelp data, but with **binary focal loss** instead of BCE +
class weights. Focal down-weights easy examples (γ) and α weights the minority fake class.

**Best-shot / tuned:** instead of hard-set γ=2 / α=0.89, this notebook **sweeps** (γ, α) —
**Phase A** tunes them on a fast proxy (a 150k stratified train subsample, capped epochs, scored
on the full balanced val set), then **Phase B** retrains from scratch at the best config on the
**full** train, full epochs. Selection by val macro-F1; chosen (γ, α) recorded in the meta.
Everything else (early stop on val macro-F1, val-tuned threshold, ONNX export) is identical to
standard Model B → serving-compatible and strictly comparable.

**Expectation (honest):** the text-only focal ablation landed *below* baseline, and focal only
reweights the loss (adds no new signal), so even tuned this may not beat the standard Model B
LSTM. The sweep gives it a genuine best shot for the ablation table.

**Before running:** upload `data/yelp_multimodal_features_full.csv` (~480 MB) to Drive root; GPU
(high-RAM) runtime. **Output:** downloads `yelp_multimodal_lstm_focal_onnx.zip` → extract to
`data/yelp_multimodal_lstm_focal_onnx/` (feature order == `model_b.FEAT_COLS`).

In [ ]:
# GloVe 300d (skip if already present)
import os
if not os.path.exists('glove.6B.300d.txt'):
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.300d.txt
print('GloVe ready:', os.path.exists('glove.6B.300d.txt'))

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

CSV     = '/content/drive/MyDrive/yelp_multimodal_features_full.csv'   # PURE full 678k (no AI)
MAX_LEN = 250
VOCAB   = 30000
EMBED   = 300
DROP    = {'user_reviews_on_this_biz'}
META    = {'text','label','split','source','fake_type','generator','length_bucket'}

df = pd.read_csv(CSV)
df['text'] = df['text'].astype(str)
feat_cols = [c for c in df.columns if c not in META and c not in DROP]
print('rows', len(df), '| behavioural features', len(feat_cols))
# fake rate per split: train is the true ~11% (imbalanced), val/test balanced 50/50
print(df.groupby('split')['label'].agg(n='size', fake_rate='mean'))

In [ ]:
# splits
tr = df[df.split=='train'].reset_index(drop=True)
va = df[df.split=='val'].reset_index(drop=True)
te = df[df.split=='test'].reset_index(drop=True)
y_tr, y_va, y_te = tr.label.values, va.label.values, te.label.values

# --- TEXT modality: tokenize + pad ---
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tok = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tok.fit_on_texts(tr.text.values)
def seq(s): return pad_sequences(tok.texts_to_sequences(s), maxlen=MAX_LEN, padding='post', truncating='post')
Xtr_t, Xva_t, Xte_t = seq(tr.text.values), seq(va.text.values), seq(te.text.values)

# --- BEHAVIOURAL modality: standardize (fit on train) ---
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
Xtr_b = sc.fit_transform(tr[feat_cols].values).astype('float32')
Xva_b = sc.transform(va[feat_cols].values).astype('float32')
Xte_b = sc.transform(te[feat_cols].values).astype('float32')
print('text', Xtr_t.shape, '| behavioural', Xtr_b.shape)

In [ ]:
# GloVe embedding matrix
emb_index = {}
with open('glove.6B.300d.txt', encoding='utf-8') as f:
    for line in f:
        v = line.split()
        emb_index[v[0]] = np.asarray(v[1:], dtype='float32')
emb_matrix = np.zeros((VOCAB, EMBED), dtype='float32')
hit = 0
for w, i in tok.word_index.items():
    if i < VOCAB:
        e = emb_index.get(w)
        if e is not None:
            emb_matrix[i] = e; hit += 1
print('GloVe coverage:', hit, '/', min(len(tok.word_index), VOCAB))

In [ ]:
# Model builder + focal loss factory + early-stopping callback + a reusable fit_once
# (fresh model per call, so the sweep and the final train share one code path).
from tensorflow.keras.layers import (Input, Embedding, Bidirectional, LSTM,
    SpatialDropout1D, Dense, Dropout, Concatenate)
from tensorflow.keras.models import Model
from sklearn.metrics import f1_score

def build_model():
    # NOTE: do NOT call keras.backend.clear_session() here. On Colab GPU (Keras 3) it invalidates
    # the CUDA handle used by the seed generator, and the next Embedding weight-init throws
    # CUDA_ERROR_INVALID_HANDLE. Each build already produces an independent model with unique
    # layer names; at most ~2 are alive at once in the sweep, so there is no leak.
    t_in = Input(shape=(MAX_LEN,), name='text_input')
    x = Embedding(VOCAB, EMBED, weights=[emb_matrix], trainable=False)(t_in)   # input_length deprecated -> removed
    x = SpatialDropout1D(0.3)(x)
    x = Bidirectional(LSTM(128))(x)
    x = Dropout(0.4)(x)
    text_repr = Dense(64, activation='relu')(x)
    b_in = Input(shape=(len(feat_cols),), name='beh_input')
    b = Dense(64, activation='relu')(b_in)
    b = Dropout(0.3)(b)
    beh_repr = Dense(32, activation='relu')(b)
    f = Concatenate()([text_repr, beh_repr])
    f = Dense(64, activation='relu')(f)
    f = Dropout(0.4)(f)
    out = Dense(1, activation='sigmoid')(f)
    return Model([t_in, b_in], out)

# ── FOCAL LOSS (replaces binary_crossentropy + class weights) ───────────────
# Binary focal loss on the sigmoid output. gamma down-weights easy examples; alpha weights the
# FAKE class (alpha=fake-class weight). BOTH are SWEPT (cell-6) — the focal paper found a LOW
# alpha for the rare class can beat a high one, so we tune rather than hard-set alpha=genuine-frac.
# No class_weight in fit (alpha already handles imbalance; that would double-weight).
def focal_loss(gamma, alpha):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        p   = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        p_t = y_true * p + (1 - y_true) * (1 - p)
        a_t = alpha * y_true + (1 - alpha) * (1 - y_true)     # alpha applied to fake(=1)
        bce = -(y_true * tf.math.log(p) + (1 - y_true) * tf.math.log(1 - p))
        return tf.reduce_mean(a_t * tf.pow(1 - p_t, gamma) * bce)
    return loss

class MacroF1ES(tf.keras.callbacks.Callback):
    """Early stop on VAL macro-F1; track the best threshold; restore best weights."""
    def __init__(self, vx, vy, patience=5):
        super().__init__(); self.vx=vx; self.vy=vy; self.patience=patience
        self.best=-1; self.best_w=None; self.best_thr=0.5; self.wait=0; self.best_ep=0; self.history=[]
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.vx, verbose=0).reshape(-1)
        bf, bt = -1, 0.5
        for t in np.linspace(0.05, 0.95, 19):
            s = f1_score(self.vy, (p>=t).astype(int), average='macro')
            if s>bf: bf, bt = s, t
        self.history.append(bf)
        if bf > self.best:
            self.best, self.best_thr = bf, bt
            self.best_w = self.model.get_weights(); self.wait = 0; self.best_ep = epoch+1
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None: self.model.set_weights(self.best_w)

def fit_once(gamma, alpha, Xt, Xb, y, epochs, patience, vx, vy, verbose=0):
    """Build+compile+train a fresh focal(gamma,alpha) model; returns (model, es_callback, history)."""
    m = build_model()
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=focal_loss(gamma, alpha), metrics=['accuracy'])
    cb = MacroF1ES(vx, vy, patience=patience)
    h = m.fit([Xt, Xb], y, validation_data=(vx, vy),
              epochs=epochs, batch_size=256, callbacks=[cb], verbose=verbose)
    return m, cb, h
print('builder + focal factory + fit_once ready')

In [ ]:
import itertools
# ============================ PHASE A — hyperparameter sweep ============================
# Tune focal (gamma, alpha) on a fast proxy (stratified train subsample + capped epochs), scored
# on the FULL balanced val set. BiLSTM epochs are cheap, so the proxy can run several epochs.
PROXY_N      = 150_000
PROXY_EPOCHS = 8
PROXY_PAT    = 2
GAMMAS = [1.0, 2.0, 3.0]            # focusing strength
ALPHAS = [0.25, 0.5, 0.75]         # fake-class weight (low can beat high)

rng = np.random.default_rng(42)
idx = rng.choice(len(y_tr), size=min(PROXY_N, len(y_tr)), replace=False)   # preserves ~11% fake
pXt, pXb, py = Xtr_t[idx], Xtr_b[idx], y_tr[idx]
print(f'PROXY sweep: {len(idx)} rows (fake rate {py.mean():.3f}) | '
      f'{len(GAMMAS)*len(ALPHAS)} configs x up to {PROXY_EPOCHS} epochs\n')

sweep=[]
for g,a in itertools.product(GAMMAS, ALPHAS):
    _, cb, _ = fit_once(g, a, pXt, pXb, py, PROXY_EPOCHS, PROXY_PAT, [Xva_t, Xva_b], y_va, verbose=0)
    sweep.append((cb.best, g, a))
    print(f'  config gamma={g} alpha={a}  ->  proxy val macroF1={cb.best:.4f} (ep {cb.best_ep})')
sweep.sort(reverse=True)
print('\nPROXY ranking (best first):')
for f1,g,a in sweep: print(f'   gamma={g} alpha={a}: {f1:.4f}')
BEST_F1, BEST_GAMMA, BEST_ALPHA = sweep[0]
print(f'\n==> BEST proxy config: gamma={BEST_GAMMA}, alpha={BEST_ALPHA} (proxy val {BEST_F1:.4f})')

# ============================ PHASE B — final full-data train ============================
# Retrain from scratch at the best config on the FULL train data, full epochs. Sets the globals
# used by the cells below: model, es, hist.
print('\n=============== FINAL: full-data train @ best config ===============')
model, es, hist = fit_once(BEST_GAMMA, BEST_ALPHA, Xtr_t, Xtr_b, y_tr,
                           30, 5, [Xva_t, Xva_b], y_va, verbose=1)
FOCAL_GAMMA, FOCAL_ALPHA = BEST_GAMMA, BEST_ALPHA   # recorded into meta on export
print(f'FINAL: best epoch {es.best_ep}  val macroF1={es.best:.4f} @thr {es.best_thr:.2f} '
      f'(gamma={FOCAL_GAMMA}, alpha={FOCAL_ALPHA})')

In [ ]:
# Training curves
import matplotlib.pyplot as plt
h=hist.history
fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].plot(h['loss'],label='train'); ax[0].plot(h['val_loss'],label='val'); ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(h['accuracy'],label='train'); ax[1].plot(h['val_accuracy'],label='val'); ax[1].set_title('Accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend()
ax[2].plot(range(1,len(es.history)+1), es.history, marker='o', color='green', label='val macro-F1')
ax[2].axvline(es.best_ep, ls='--', color='red', label=f'best epoch {es.best_ep}')
ax[2].set_title('Val macro-F1'); ax[2].set_xlabel('epoch'); ax[2].legend()
plt.tight_layout(); plt.savefig('yelp_multimodal_lstm_training.png', dpi=120); plt.show()

In [ ]:
# Test evaluation (val-tuned threshold) — pure human-fake task, no per-AI-type breakdown
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, classification_report
thr  = es.best_thr
p_te = model.predict([Xte_t, Xte_b], verbose=0).reshape(-1)
pred = (p_te >= thr).astype(int)
print('threshold (val-tuned macroF1) =', round(thr, 2))
print('test accuracy  =', round(accuracy_score(y_te, pred), 4))
print('test macro-F1  =', round(f1_score(y_te, pred, average='macro'), 4))
print('fake recall    =', round(recall_score(y_te, pred, pos_label=1), 3))
print('fake precision =', round(precision_score(y_te, pred, pos_label=1), 3))
print('genuine spec   =', round(recall_score(y_te, pred, pos_label=0), 3))
print('\nReferences (same Yelp test): text-only ~69% · pure sklearn Model B (full) ens 76.2% / xgb 77.3%')
print()
print(classification_report(y_te, pred, target_names=['genuine','fake'], digits=4))

In [ ]:
# Save artifacts (keras for CPU re-export, + scaler/tokenizer/threshold for serving)
import json, joblib
THR=float(es.best_thr)
model.save('yelp_multimodal_lstm.keras')
joblib.dump(sc, 'yelp_multimodal_lstm_scaler.joblib')          # serving must standardise behaviour
with open('yelp_multimodal_lstm_tokenizer.json','w',encoding='utf-8') as f: f.write(tok.to_json())
json.dump({'threshold':THR,'feat_cols':feat_cols,'max_len':MAX_LEN,'vocab':VOCAB,
           'focal_gamma':float(FOCAL_GAMMA),'focal_alpha':float(FOCAL_ALPHA)},
          open('yelp_multimodal_lstm_meta.json','w'), indent=2)
print('saved keras + scaler + tokenizer + meta | threshold', THR, '| focal', FOCAL_GAMMA, FOCAL_ALPHA)

In [ ]:
# Export to ONNX (thread-safe serving, like the project's other DL models).
# Re-export the SavedModel with CUDA hidden so TF uses standard LSTM ops, not
# CudnnRNNV3 (which ONNX Runtime can't run). Multi-input: text_input + beh_input.
# compile=False on load -> don't need the custom focal loss for inference/export.
!pip install -q tf2onnx onnxruntime
import subprocess, os
open('cpu_export.py','w').write(
'''import os
os.environ["CUDA_VISIBLE_DEVICES"]=""
import tensorflow as tf
m=tf.keras.models.load_model("yelp_multimodal_lstm.keras", compile=False)
m.export("yelp_multimodal_lstm_savedmodel")
print("CPU SavedModel export done")
''')
r1=subprocess.run(['python','cpu_export.py'],capture_output=True,text=True,
                  env={**os.environ,'CUDA_VISIBLE_DEVICES':''})
print(r1.stdout);  print('STDERR:',r1.stderr[-800:]) if r1.returncode else None
r2=subprocess.run(['python','-m','tf2onnx.convert','--saved-model','yelp_multimodal_lstm_savedmodel',
                   '--output','yelp_multimodal_lstm.onnx','--opset','13'],capture_output=True,text=True)
print(r2.stdout[-400:]); print('STDERR:',r2.stderr[-1200:]) if r2.returncode else None

In [ ]:
# Sanity check: ONNX outputs must match Keras (feed both inputs, mapped by last dim)
import onnxruntime as ort, numpy as np
sess=ort.InferenceSession('yelp_multimodal_lstm.onnx', providers=['CPUExecutionProvider'])
print('ONNX inputs:', [(i.name,i.shape) for i in sess.get_inputs()])
feed={}
for i in sess.get_inputs():
    feed[i.name]=(Xte_t[:4].astype(np.float32) if i.shape[-1]==MAX_LEN else Xte_b[:4].astype(np.float32))
onnx_p=sess.run(None, feed)[0].flatten()
keras_p=model.predict([Xte_t[:4],Xte_b[:4]],verbose=0).flatten()
print('ONNX :', np.round(onnx_p,4))
print('Keras:', np.round(keras_p,4))
print('match:', bool(np.allclose(onnx_p,keras_p,atol=1e-3)))

In [ ]:
# Bundle + download -> extract to data/yelp_multimodal_lstm_focal_onnx/
import zipfile
from google.colab import files
with zipfile.ZipFile('yelp_multimodal_lstm_focal_onnx.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for fn in ['yelp_multimodal_lstm.onnx','yelp_multimodal_lstm_tokenizer.json',
               'yelp_multimodal_lstm_scaler.joblib','yelp_multimodal_lstm_meta.json']:
        zf.write(fn)
print('created yelp_multimodal_lstm_focal_onnx.zip  (extract -> data/yelp_multimodal_lstm_focal_onnx/)')
files.download('yelp_multimodal_lstm_focal_onnx.zip')